In [1]:
print("all ok")

all ok


In [2]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()

/Users/arunkota/Documents/MLOPS/document_portal/env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [3]:
model=ChatGroq(model="qwen/qwen3.6-27b")

In [4]:
model.invoke("Hello, how are you?")

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User says: "Hello, how are you?"\n   - This is a standard greeting and a polite inquiry about my state.\n\n2.  **Identify Key Components:**\n   - Greeting: "Hello"\n   - Question: "how are you?"\n   - Tone: Friendly, conversational\n\n3.  **Determine Response Strategy:**\n   - Acknowledge the greeting\n   - Respond to the question politely\n   - Keep it concise and friendly\n   - Offer assistance (standard AI behavior)\n   - Maintain appropriate tone (warm, professional)\n\n4.  **Draft Response (Mental):**\n   Hello! I\'m doing well, thank you for asking. How can I help you today?\n\n5.  **Refine Response:**\n   - Check for tone: Friendly and professional ✓\n   - Check for completeness: Answers the question, returns the courtesy, offers help ✓\n   - Check for accuracy: As an AI, I don\'t have feelings, but it\'s standard and polite to say I\'m doing well/functioning properly. I\'ll keep it na

In [5]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings

In [6]:
# embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7668.38it/s]


In [7]:
embeddings.embed_query("what is the capital of France?")

[0.08204811811447144,
 0.03605549409985542,
 -0.003892886219546199,
 -0.004881037399172783,
 0.02565111592411995,
 -0.05714346468448639,
 0.012191607616841793,
 0.00467892549932003,
 0.034949857741594315,
 -0.022421933710575104,
 -0.00800525676459074,
 -0.10935357958078384,
 0.022724764421582222,
 -0.029320841655135155,
 -0.04352204129099846,
 -0.1202412024140358,
 -0.0008485977305099368,
 -0.018150124698877335,
 0.056129567325115204,
 0.0030852709896862507,
 0.002336367266252637,
 -0.016839269548654556,
 0.06362468749284744,
 -0.023660238832235336,
 0.03149351477622986,
 -0.034797947853803635,
 -0.020548803731799126,
 -0.0027909809723496437,
 -0.011038010939955711,
 -0.036126717925071716,
 0.054141074419021606,
 -0.036617156118154526,
 -0.02500869333744049,
 -0.038170408457517624,
 -0.04960362985730171,
 -0.0151481032371521,
 0.021315041929483414,
 -0.012740438804030418,
 0.07670089602470398,
 0.044355764985084534,
 -0.010834875516593456,
 -0.029760010540485382,
 -0.016970476135611534

In [8]:
# Data Ingestion
from langchain_community.document_loaders import PyPDFLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter



/var/folders/6d/3_jhlz_d467_p_7hxyp4wx780000gn/T/ipykernel_63556/3336914083.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [9]:
import os

In [10]:
file_path=os.path.join(os.getcwd(), "data", "data.pdf")
loader =PyPDFLoader(file_path)
documents=loader.load()
documents


[Document(metadata={'producer': 'calibre 3.42.0 [https://calibre-ebook.com]', 'creator': 'calibre 3.42.0 [https://calibre-ebook.com]', 'creationdate': '2019-07-26T17:39:48+00:00', 'author': 'Marshall Goldsmith & Mark Reiter', 'keywords': 'Finance, Business', 'title': "What got you here won\\'t get you there : how successful people become even more successful - PDFDrive.com", 'source': '/Users/arunkota/Documents/MLOPS/document_portal/notebook/data/data.pdf', 'total_pages': 234, 'page': 0, 'page_label': '1'}, page_content=''),
 Document(metadata={'producer': 'calibre 3.42.0 [https://calibre-ebook.com]', 'creator': 'calibre 3.42.0 [https://calibre-ebook.com]', 'creationdate': '2019-07-26T17:39:48+00:00', 'author': 'Marshall Goldsmith & Mark Reiter', 'keywords': 'Finance, Business', 'title': "What got you here won\\'t get you there : how successful people become even more successful - PDFDrive.com", 'source': '/Users/arunkota/Documents/MLOPS/document_portal/notebook/data/data.pdf', 'total_

In [11]:
len(documents)

234

In [12]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100000, chunk_overlap=0, length_function=len)

In [13]:
docs = text_splitter.split_documents(documents)
len(docs)

233

In [15]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(docs, embeddings)

In [24]:
relavant_doc =vectorstore.similarity_search("what is the capital of france", k=2)

In [27]:
retriever = vectorstore.as_retriever()

In [28]:
prompt_template = """Use the following pieces of context to answer the question at the end.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
Context : {context} 
Question: {question}
Answer in Markdown:"""  

In [ ]:
from langchain.prompts import PromptTemplate
PromptTemplate = PromptTemplate(
    template=prompt_template, input_variables=["context", "question"]
)